In [0]:
df_movies = spark.read.csv("/Volumes/workspace/default/moviesdata/movies.csv", header=True, inferSchema=True)
df_ratings = spark.read.csv("/Volumes/workspace/default/moviesdata/ratings.csv", header=True, inferSchema=True)
df_tags = spark.read.csv("/Volumes/workspace/default/moviesdata/tags.csv", header=True, inferSchema=True)
df_links = spark.read.csv("/Volumes/workspace/default/moviesdata/links.csv", header=True, inferSchema=True)

In [0]:
from pyspark.sql.functions import when, col

df_movies_null = df_movies.withColumn(
    "title",
    when(col("movieId") % 10 == 0, None).otherwise(col("title"))
).withColumn(
    "genres",
    when(col("movieId") % 15 == 0, None).otherwise(col("genres"))
)

In [0]:
df_movies_null.display()

In [0]:
df_ratings_null = df_ratings.withColumn(
    "rating",
    when(col("rating") % 3 == 0, None).otherwise(col("rating"))
)

In [0]:
df_ratings_null.display()

In [0]:
df_ratings_dup = df_ratings_null.union(df_ratings_null.limit(100))

In [0]:
df_ratings_dup.display()


**checking duplicate rows**


In [0]:
df_ratings_dup.groupBy("userId", "movieId", "rating", "timestamp") \
    .count() \
    .filter("count > 1") \
    .show()

In [0]:
df_movies_dup = df_movies_null.union(df_movies_null.limit(50))

 **# invalid rating (>5)**

In [0]:
from pyspark.sql.functions import lit

df_ratings_dirty = df_ratings_dup.withColumn(
    "rating",
    when(col("userId") % 30 == 0, 7.5)  # invalid rating (>5)
    .otherwise(col("rating"))
)

**Wrong formats (string instead of number)**

In [0]:
df_links_dirty = df_links.withColumn(
    "imdbId",
    when(col("movieId") % 20 == 0, "unknown").otherwise(col("imdbId"))
)

**for saving dirty datasets**

In [0]:
dbutils.fs.rm("/Volumes/workspace/default/moviesdata", True)


In [0]:
dbutils.fs.rm("/Volumes/workspace/default/moviesdata", True)

In [0]:
df_movies_dup.write.mode("overwrite").format("delta") \
    .save("/Volumes/workspace/default/moviesdirtydata")